## Install Faiss

In [1]:
! pip install faiss-cpu

In [2]:
import faiss

print(faiss.__version__)

1.13.2


### Step 2: Load Saved Embeddings

In [3]:
import pickle

with open(
    "../models/nvidia_embeddings.pkl",
    "rb"
) as f:

    embeddings = pickle.load(f)

print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(4, 384)


### Step 3: Convert to Float32


In [4]:
import numpy as np

embeddings = np.array(
    embeddings,
    dtype=np.float32
)

print(embeddings.dtype)

float32


### Create faiss index

In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001D021B70B40> >


### Step 5: Add Embeddings

In [6]:
index.add(
    embeddings
)

print(
    "Total vectors:",
    index.ntotal
)

Total vectors: 4


## Save Index

In [7]:
faiss.write_index(
    index,
    "../models/nvidia_faiss.index"
)

### load Model

In [9]:
index = faiss.read_index(
    "../models/nvidia_faiss.index"
)

### Step 8: Search

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

C:\Users\Suryaprakash\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
query = "AI hardware"

In [12]:
query_vector = model.encode(
    [query]
)

query_vector = np.array(
    query_vector,
    dtype=np.float32
)

### retreive top 5 

In [13]:
k = 5

distances, indices = index.search(
    query_vector,
    k
)

print(indices)

[[ 0  1  2  3 -1]]


### Show Results

In [18]:
with open(
    "../data/raw/sample_text.txt",
    "r",
    encoding="utf-8"
) as file:

    sentences = [
        line.strip()
        for line in file.readlines()
        if line.strip()
    ]

print("Total Sentences:", len(sentences))

Total Sentences: 100


In [19]:
sentences

['The NVIDIA GeForce RTX platform helps organizations scale data science workloads.',
 'The NVIDIA CUDA platform helps organizations scale machine learning workloads.',
 'Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.',
 'robotics solutions benefit from the performance of NVIDIA Blackwell.',
 'Researchers rely on NVIDIA DGX to build advanced autonomous vehicles systems.',
 'The NVIDIA CUDA platform helps organizations scale machine learning workloads.',
 'robotics solutions benefit from the performance of NVIDIA Omniverse.',
 'robotics solutions benefit from the performance of NVIDIA CUDA.',
 'Developers use NVIDIA DGX for autonomous vehicles projects across industries.',
 'Researchers rely on NVIDIA GeForce RTX to build advanced data science systems.',
 'Researchers rely on NVIDIA Omniverse to build advanced computer vision systems.',
 'Developers use NVIDIA DGX for autonomous vehicles projects across industries.',
 'Researchers rely on NVIDIA Omni

In [20]:
for idx in indices[0]:

    print(
        sentences[idx]
    )

The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
The NVIDIA CUDA platform helps organizations scale machine learning workloads.
Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
robotics solutions benefit from the performance of NVIDIA Blackwell.
NVIDIA TensorRT advances autonomous vehicles applications in modern computing.


In [21]:
query = "artificial intelligence"

In [22]:
distances, indices = index.search(
    query_vector,
    k=5
)

In [23]:
print(query)

artificial intelligence


### Reusable Function

In [24]:
def search(query, k=5):

    query_vector = model.encode([query])

    query_vector = np.array(
        query_vector,
        dtype=np.float32
    )

    distances, indices = index.search(
        query_vector,
        k
    )

    print(f"\nQuery: {query}\n")

    for i, idx in enumerate(indices[0], 1):

        print(f"{i}. {sentences[idx]}")

In [25]:
search("artificial intelligence")

search("machine learning")

search("gaming")

search("deep learning")


Query: artificial intelligence

1. robotics solutions benefit from the performance of NVIDIA Blackwell.
2. The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
3. The NVIDIA CUDA platform helps organizations scale machine learning workloads.
4. Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
5. NVIDIA TensorRT advances autonomous vehicles applications in modern computing.

Query: machine learning

1. The NVIDIA CUDA platform helps organizations scale machine learning workloads.
2. robotics solutions benefit from the performance of NVIDIA Blackwell.
3. The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
4. Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
5. NVIDIA TensorRT advances autonomous vehicles applications in modern computing.

Query: gaming

1. Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
2. The NVIDIA GeForce 

### Create Retrieve Function

In [26]:
def retrieve(query, k=5):

    query_vector = model.encode([query])

    query_vector = np.array(
        query_vector,
        dtype=np.float32
    )

    distances, indices = index.search(
        query_vector,
        k
    )

    results = []

    for idx in indices[0]:
        results.append(sentences[idx])

    return results

In [27]:
retrieve("artificial intelligence")

['robotics solutions benefit from the performance of NVIDIA Blackwell.',
 'The NVIDIA GeForce RTX platform helps organizations scale data science workloads.',
 'The NVIDIA CUDA platform helps organizations scale machine learning workloads.',
 'Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.',
 'NVIDIA TensorRT advances autonomous vehicles applications in modern computing.']

In [28]:
query = "artificial intelligence"

results = retrieve(query)

context = "\n".join(results)

print(context)

robotics solutions benefit from the performance of NVIDIA Blackwell.
The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
The NVIDIA CUDA platform helps organizations scale machine learning workloads.
Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
NVIDIA TensorRT advances autonomous vehicles applications in modern computing.


In [29]:
def answer(query):

    results = retrieve(query)

    context = "\n".join(results)

    return f"""
Question:
{query}

Context:
{context}
"""

### Testing the MINI RAG

In [30]:
print(
    answer(
        "artificial intelligence"
    )
)


Question:
artificial intelligence

Context:
robotics solutions benefit from the performance of NVIDIA Blackwell.
The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
The NVIDIA CUDA platform helps organizations scale machine learning workloads.
Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
NVIDIA TensorRT advances autonomous vehicles applications in modern computing.



In [31]:
query = input("Ask a question: ")

print(answer(query))


Question:
AI

Context:
The NVIDIA GeForce RTX platform helps organizations scale data science workloads.
robotics solutions benefit from the performance of NVIDIA Blackwell.
The NVIDIA CUDA platform helps organizations scale machine learning workloads.
Researchers rely on NVIDIA Blackwell to build advanced autonomous vehicles systems.
NVIDIA TensorRT advances autonomous vehicles applications in modern computing.

